In this tutorial, I will show you the differences and similarities between [TLS](https://github.com/hippke/tls) and GTLS. Please noticed that GTLS can only run on a computer with a CUDA GPU. And the speed is highly variable depending on your GPU.

This tutorial is adapted from Michael Hippke 's excellent [TLS tutorial](https://github.com/hippke/tls/tree/master/tutorials).

At first, I will use a TESS Target.

In [ ]:
import matplotlib.pyplot as plt; plt.rcParams["figure.dpi"] = 150
import astropy
import numpy
from astropy.io import fits
from astropy.stats import sigma_clip

from transitleastsquares import (
    transitleastsquares,
    cleaned_array,
    catalog_info,
    transit_mask
    )

# We find the URLs of the FITS files on this website:
# https://archive.stsci.edu/prepds/tess-data-alerts/
url = 'https://archive.stsci.edu/hlsps/tess-data-alerts/hlsp_tess-data-alerts_tess_phot_00159951311-s03_tess_v1_lc.fits'
hdu = fits.open(url)
time = hdu[1].data['TIME']
flux = hdu[1].data['PDCSAP_FLUX']  # values with non-zero quality are nan or zero'ed
time, flux = cleaned_array(time, flux)  # remove invalid values such as nan, inf, non, negative
flux = flux / numpy.median(flux)

plt.figure()
plt.xlabel('Time (days)')
plt.ylabel('flux (e-/s)')
plt.scatter(time, flux, color='black', s=1, alpha=0.5);

In [6]:
print(hdu[1].header)

SIMPLE  =                    T / conforms to FITS standards                     BITPIX  =                    8 / array data type                                NAXIS   =                    0 / number of array dimensions                     EXTEND  =                    T / file contains extensions                       NEXTEND =                    2 / number of standard extensions                  EXTNAME = 'PRIMARY '           / name of extension                              EXTVER  =                    1 / extension version number (not format version)  SIMDATA =                    F / file is based on simulated data                ORIGIN  = 'NASA/Ames'          / institution responsible for creating this file DATE    = '2018-11-24'         / file creation date.                            TSTART  =    1382.033915694891 / observation start time in TJD                  TSTOP   =    1409.383908993930 / observation stop time in TJD                   DATE-OBS= '2018-09-20T12:47:41.132Z' / T

In [ ]:
TIC_ID = hdu[0].header['TICID']
ab, mass, mass_min, mass_max, radius, radius_min, radius_max = catalog_info(TIC_ID=TIC_ID)
print('Searching with limb-darkening estimates using quadratic LD (a,b)=', ab)

In [ ]:
import time as Nowtime
TLS_start = Nowtime.time()
model = transitleastsquares(time, flux)
results = model.power()
print('period', results.period, 'duration', results.duration, 'depth', results.depth, 'T0', results.T0,'SDE', results.SDE)
print('TLS took',Nowtime.time() - TLS_start,'seconds')

In [ ]:
from gputls import gtls
GTLS_start = Nowtime.time()
model = gtls(t = time, y = flux)
gtlsResult = model.power()
print('period', gtlsResult.period, 'duration', gtlsResult.duration, 'depth', gtlsResult.depth, 'T0', gtlsResult.T0,'SDE', gtlsResult.SDE)
print('GTLS took',Nowtime.time() - GTLS_start,'seconds')

As you might see, GTLS basically produces the same results as TLS.

But GTLS is faster than TLS. The sepcific speed is highly variable depending on your GPU.

I use a RTX3060 GPU, a very cheap model for exhibition, and the speed is about 5-6 times faster than my Ryzen R5 3600 CPU.